In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Configuration

from src.io.tickers import load_ticker_dictionary, build_ticker_type_map, DictionaryColumns
from src.io.artifacts import save_json, make_run_id
from src.config import Config

conf = Config(
    markets=["SPX Index"],  # add more markets as needed
    dictionary_path="data/input/230210 Factor Set - Dictionnary (bbg only) - simplified.xlsx",
    returns_path="data/input/230216_returns_simplified.csv",
)

run_id = make_run_id(conf.hash(), label="sri_2000_2009_simplified")

dic = load_ticker_dictionary(conf.dictionary_path, DictionaryColumns(symbol="SYMBOL", nature="Nature"))
ticker_map = build_ticker_type_map(dic)

save_json(ticker_map, f"{conf.artifacts_root}/{run_id}/ticker_map.json")
save_json(conf.to_dict(), f"{conf.artifacts_root}/{run_id}/config.json")

c:\Users\sylva\Documents\Python\Polymodels\src\io\tickers.py:59: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_rate = nature.str.contains(rate_regex, regex=True, na=False)


In [3]:
# Load and validate returns

from src.io.loaders import load_returns, validate_returns
from src.io.artifacts import save_parquet

rets = load_returns(conf.returns_path)
validate_returns(rets)

save_parquet(rets, f"{conf.artifacts_root}/{run_id}/returns_daily_raw.parquet")

In [4]:
# Preprocess returns

from src.preprocessing.clipping import basic_nan_cleanup, clip_returns_by_type
from src.io.artifacts import save_parquet, load_json

ticker_map = load_json(f"{conf.artifacts_root}/{run_id}/ticker_map.json")

rets = basic_nan_cleanup(rets)
if conf.is_clipping:
    rets_clean = clip_returns_by_type(
    rets,
    ticker_map=ticker_map,
    asset_threshold=conf.clip_assets,
    rate_threshold=conf.clip_rates,
    mode=conf.clipping_mode,
)
else:
    rets_clean = rets
save_parquet(rets_clean, f"{conf.artifacts_root}/{run_id}/returns_daily_clean.parquet")

In [5]:
# Compute 21-day rolling compounded features

from src.features.rolling_21d import compute_21d_compounded_features
from src.io.artifacts import save_parquet

features_21d_daily = compute_21d_compounded_features(
    returns_df=rets_clean,
    ticker_type_map=ticker_map,
    window=conf.rolling_window_days,
)
save_parquet(features_21d_daily, f"{conf.artifacts_root}/{run_id}/features_21d_daily.parquet")

c:\Users\sylva\Documents\Python\Polymodels\src\features\rolling_21d.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = compounded_rolling_return(s, window=window)
c:\Users\sylva\Documents\Python\Polymodels\src\features\rolling_21d.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = compounded_rolling_return(s, window=window)
c:\Users\sylva\Documents\Python\Polymodels\src\features\rolling_21d.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` m

In [30]:
# Extract last trading days of the month
from src.io.artifacts import save_parquet, load_parquet
import pandas as pd

features_21d_daily = load_parquet(f"{conf.artifacts_root}/{run_id}/features_21d_daily.parquet")
daily_idx = features_21d_daily.index

month_ends_idx = pd.date_range(
    start=features_21d_daily.index.min(),
    end=features_21d_daily.index.max(),
    freq='ME'
    )

month_ends_df = month_ends_idx.to_frame(index=False, name="month_end")
save_parquet(month_ends_df, f"{conf.artifacts_root}/{run_id}/month_ends.parquet")

print(f"Extracted {len(month_ends_df)} actual trading month-ends.")

Extracted 83 actual trading month-ends.


In [31]:
# Build and align X and Y daily datasets

from src.features.alignment import XYSpec, build_XY_monthly, apply_daily_lag, align_XY_after_lag
from src.io.artifacts import save_parquet, load_parquet

features_21d_daily = load_parquet(f"{conf.artifacts_root}/{run_id}/features_21d_daily.parquet")

ticker_map = load_json(f"{conf.artifacts_root}/{run_id}/ticker_map.json")
spec = XYSpec(markets=conf.markets, factors=[c for c in ticker_map if c not in conf.markets])
Y_daily, X_daily = build_XY_monthly(features_21d_daily, spec)

X_lagged_daily = apply_daily_lag(X_daily, lag_days=21)
Y_aligned, X_aligned = align_XY_after_lag(Y_daily, X_lagged_daily)

save_parquet(Y_aligned, f"{conf.artifacts_root}/{run_id}/Y_daily_aligned.parquet")
save_parquet(X_aligned, f"{conf.artifacts_root}/{run_id}/X_daily_lagged_aligned.parquet")

In [16]:
# LNLM panel estimation
from src.models.estimation_panel import run_lnlm_panel_estimation, EstimationParams
from src.io.artifacts import save_parquet, load_parquet
import pandas as pd

Y_aligned = load_parquet(f"{conf.artifacts_root}/{run_id}/Y_daily_aligned.parquet")
X_lagged = load_parquet(f"{conf.artifacts_root}/{run_id}/X_daily_lagged_aligned.parquet")
month_ends_df = load_parquet(f"{conf.artifacts_root}/{run_id}/month_ends.parquet")

target_dates = pd.DatetimeIndex(month_ends_df["month_end"])

print(f"Dimensions Y : {Y_aligned.shape}")
print(f"Dimensions X : {X_lagged.shape}")
common_idx = Y_aligned.index.intersection(X_lagged.index)
valid_dates = [t for t in target_dates if t in common_idx]
print(f"Out of {len(target_dates)} target dates, {len(valid_dates)} exist in the common index X/Y.")

est_params = EstimationParams(
    window_days=1260,          
    min_obs_per_model=252,     
    degree_n=4,
    n_folds=5,
    n_mu_points=100,
    random_state=42,
)

rmse_long, params_long = run_lnlm_panel_estimation(
    Y_daily=Y_aligned, 
    X_lagged_daily=X_lagged, 
    target_dates=target_dates, 
    params=est_params
)

print(f"Number of successful LNLM models : {len(rmse_long)}")

save_parquet(rmse_long, f"{conf.artifacts_root}/{run_id}/rmse_panel_long.parquet")
save_parquet(params_long, f"{conf.models_root   }/{run_id}/lnlm_params_long.parquet")

Dimensions Y : (678, 1)
Dimensions X : (678, 2)
Out of 84 target dates, 80 exist in the common index X/Y.
Number of successful LNLM models : 89


In [20]:
# Build RMSE distributions with KDE

import numpy as np
from src.io.artifacts import load_parquet, save_parquet, save_npy
from src.distributions.kde import KDEParams, build_global_grid_from_rmse, build_Pt_distributions

rmse_long = load_parquet(f"{conf.artifacts_root}/{run_id}/rmse_panel_long.parquet")

grid = build_global_grid_from_rmse(rmse_long, grid_size=conf.kde_grid_size)
save_npy(grid, f"{conf.artifacts_root}/{run_id}/rmse_grid.npy")

kde_params = KDEParams(
    grid_size=conf.kde_grid_size,
    bandwidth=conf.kde_bandwidth,
    keep_top_quantile=1.0,  # configurable, normally 0.8
    min_samples=1 # 10
)

Pt = build_Pt_distributions(rmse_long, grid=grid, kde_params=kde_params)
save_parquet(Pt, f"{conf.artifacts_root}/{run_id}/Pt_distributions.parquet")

In [21]:
# Build memory-adjusted distributions Qt and Rt

import pandas as pd
from src.io.artifacts import load_parquet, save_parquet, load_npy
from src.distributions.memory import MemoryParams, forward_cumulative_return, build_memory_distributions

grid = load_npy(f"{conf.artifacts_root}/{run_id}/rmse_grid.npy")
Pt = load_parquet(f"{conf.artifacts_root}/{run_id}/Pt_distributions.parquet")

Y_daily = load_parquet(f"{conf.artifacts_root}/{run_id}/Y_daily_aligned.parquet")
month_ends_df = load_parquet(f"{conf.artifacts_root}/{run_id}/month_ends.parquet")

target_dates = pd.DatetimeIndex(month_ends_df["month_end"])
common_dates = target_dates.intersection(Y_daily.index)
Y_monthly = Y_daily.loc[common_dates].copy()

mem_params = MemoryParams(
    forward_months=conf.forward_months,
    halflife_years=conf.memory_halflife_years,
    lag_months=conf.forward_months  # anti look-ahead, lag = horizon
)

Qt_list = []
Rt_list = []

for mkt in conf.markets:
    # Forward 3M return of this market (calcule maintenant bien 3 mois, et non 3 jours)
    fwd_3m = forward_cumulative_return(Y_monthly[mkt], horizon_months=mem_params.forward_months)

    # Subset Pt for this market
    Pt_m = Pt.xs(mkt, level="market", drop_level=False)

    Qt_m, Rt_m = build_memory_distributions(
        Pt=Pt_m,
        grid=grid,
        market_forward_3m=fwd_3m,
        mem_params=mem_params
    )

    # Store as wide but with market in MultiIndex for consistency
    Qt_m.index.name = "date"
    Rt_m.index.name = "date"

    Qt_m["market"] = mkt
    Rt_m["market"] = mkt

    Qt_list.append(Qt_m.reset_index().set_index(["date", "market"]))
    Rt_list.append(Rt_m.reset_index().set_index(["date", "market"]))

Qt = pd.concat(Qt_list).sort_index()
Rt = pd.concat(Rt_list).sort_index()

save_parquet(Qt, f"{conf.artifacts_root}/{run_id}/Qt_precrisis.parquet")
save_parquet(Rt, f"{conf.artifacts_root}/{run_id}/Rt_normal.parquet")

print(f"Distributions Qt et Rt generated for {len(Qt)} months.")

Distributions Qt et Rt generated for 50 months.


In [22]:
# Compute Hellinger distance and SRI signal

from src.io.artifacts import load_parquet, save_parquet, load_npy
from src.distributions.hellinger import compute_hellinger_series
from src.signals.sri import sri_from_hellinger, delta_hellinger

grid = load_npy(f"{conf.artifacts_root}/{run_id}/rmse_grid.npy")
Pt = load_parquet(f"{conf.artifacts_root}/{run_id}/Pt_distributions.parquet")
Qt = load_parquet(f"{conf.artifacts_root}/{run_id}/Qt_precrisis.parquet")

H_all = []
S_all = []
dH_all = []

for mkt in conf.markets:
    Pt_m = Pt.xs(mkt, level="market", drop_level=True)
    Qt_m = Qt.xs(mkt, level="market", drop_level=True)

    H = compute_hellinger_series(Pt_m, Qt_m, grid)
    dH = delta_hellinger(H)
    sri = sri_from_hellinger(H)

    H_all.append(H.to_frame(name="hellinger").assign(market=mkt).reset_index().set_index(["date", "market"]))
    dH_all.append(dH.to_frame(name="delta_hellinger").assign(market=mkt).reset_index().set_index(["date", "market"]))
    S_all.append(sri.to_frame(name="sri").assign(market=mkt).reset_index().set_index(["date", "market"]))

H_df = pd.concat(H_all).sort_index()
dH_df = pd.concat(dH_all).sort_index()
S_df = pd.concat(S_all).sort_index()

save_parquet(H_df, f"{conf.results_root}/{run_id}/hellinger.parquet")
save_parquet(dH_df, f"{conf.results_root}/{run_id}/delta_hellinger.parquet")
save_parquet(S_df, f"{conf.results_root}/{run_id}/sri_signal.parquet")

In [24]:
# Monthly backtest

import pandas as pd
from src.io.artifacts import load_parquet, save_parquet, save_json
from src.backtest.monthly import BacktestParams, run_monthly_backtest_multi_market
from src.backtest.metrics import MetricsParams, perf_metrics_multi_market

# Inputs
returns_daily_raw = load_parquet(f"{conf.artifacts_root}/{run_id}/returns_daily_raw.parquet")
sri_df = load_parquet(f"{conf.results_root}/{run_id}/sri_signal.parquet")

bt_params = BacktestParams(
    leverage_risk_on=conf.leverage_risk_on,
    leverage_risk_off=conf.leverage_risk_off,
    risk_off_when_sri_is_one=True,     # change to False if your convention differs
    default_leverage=1.0,
    fill_missing_daily_returns_with_zero=True,
)

bt = run_monthly_backtest_multi_market(
    daily_returns_df=returns_daily_raw,
    sri_df=sri_df,
    markets=conf.markets,
    params=bt_params,
)

# Save outputs
pos_monthly = bt["monthly_positions"].copy()
pos_monthly.index = pos_monthly.index.to_timestamp(how='end')  # Converts "2008-01" in "2008-01-31"

save_parquet(pos_monthly, f"{conf.results_root}/{run_id}/positions_monthly.parquet")

save_parquet(bt["daily_leverage"], f"{conf.results_root}/{run_id}/leverage_daily.parquet")
save_parquet(bt["strategy_daily_returns"], f"{conf.results_root}/{run_id}/strategy_daily_returns.parquet")
save_parquet(bt["equity_curves"], f"{conf.results_root}/{run_id}/equity_curves.parquet")
save_parquet(bt["buyhold_equity_curves"], f"{conf.results_root}/{run_id}/buyhold_equity_curves.parquet")

# Metrics
m_params = MetricsParams(trading_days_per_year=252, risk_free_rate_annual=0.0)
metrics = perf_metrics_multi_market(bt["strategy_daily_returns"], bt["equity_curves"], params=m_params)
save_json(metrics, f"{conf.results_root}/{run_id}/perf_metrics.json")